<a href="https://colab.research.google.com/github/christy5165/Denoising_Autoencoder.ipynb/blob/main/GAN-WK-11.A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# 1. Install required libraries
!pip install -q transformers[torch] datasets

import torch
import os
import shutil
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset

# 2. Prepare Dialogue Dataset
# Format: [Scenario] [Character A] [Character B]
dialogue_data = [
    {"text": "Scenario: Lost in a city. A: I have no idea where we are. B: Just look at the map, we'll find our way eventually."},
    {"text": "Scenario: Winning a game. A: We actually did it! B: I never doubted us for a second, what a match!"},
    {"text": "Scenario: A broken car on a rainy day. A: Of course the engine dies now, in the middle of a downpour! B: Calm down, I'm calling the tow truck right now."},
    {"text": "Scenario: Cooking disaster. A: The cake is completely burnt! B: It's okay, we can just order pizza instead."}
]

raw_dataset = Dataset.from_dict({"text": [item["text"] for item in dialogue_data]})

# 3. Load Model and Tokenizer
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 4. Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = raw_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Setup and Train
output_dir = "./gpt2-dialogue"
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

model = GPT2LMHeadModel.from_pretrained(model_name)
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=40, # High epochs so the small model 'memorizes' the dialogue structure
    per_device_train_batch_size=1,
    save_strategy="no",
    logging_steps=5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

print("Conditioning model for dialogue generation...")
trainer.train()

# 6. Generate for the specific scenario
# We provide the scenario and the first character's name to trigger the dialogue
prompt = "Scenario: Character A is frustrated because their car broke down on a rainy day. A:"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)

output = model.generate(
    input_ids,
    max_length=80,
    num_return_sequences=1,
    no_repeat_ngram_size=3,
    repetition_penalty=1.2,
    temperature=0.9, # Higher temperature for more natural/varied speech
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

print("\n--- GENERATED DIALOGUE ---")
print(tokenizer.decode(output[0], skip_special_tokens=True))

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Conditioning model for dialogue generation...


Step,Training Loss
5,3.286594
10,2.158968
15,1.476866
20,1.020192
25,0.688124
30,0.555726
35,0.257903
40,0.263111
45,0.329880
50,0.166994



--- GENERATED DIALOGUE ---
Scenario: Character A is frustrated because their car broke down on a rainy day. A: Of course the engine dies now, in our situation! B- We have no idea what's going wrong—just look at us for awhile… Cough…. I'm calling my dad right here instead of letting him go ahead and call me? Duh……..F*ck off already….." —
